In [ ]:
import jax
from jax import lax, Array
import jax.numpy as jnp
from jax.scipy.linalg import expm
jax.config.update('jax_enable_x64', True)
import numpy as np
from numpy.linalg import matrix_power
from scipy.stats import entropy, poisson
from scipy.optimize import minimize
from scipy.special import softmax, logsumexp
from itertools import product as iproduct
from time import perf_counter
from functools import partial

import os, sys, pathlib
os.environ["OPENBLAS_NUM_THREADS"] = "1"
_project_root = pathlib.Path.cwd().resolve()
sys.path.insert(0, str(_project_root.parents[0]))
from utils import H, Hv, KL, KLv, get_traj
import ising
from ising import (
  propagate_ctmc, EP_cost,
  precompute_ep, make_uniformization_coeffs, pmmc_kernel_fast, pmmc_kernel_old,
  find_optimal_prior, build_system_from_graph, precompute_ep_jax, apply_G
)

GraphT = tuple[Array, Array, Array]

In [ ]:
import shutil, matplotlib as mpl
print(shutil.which("latex"))
print(shutil.which("kpsewhich"))
print(mpl.get_configdir())
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from matplotlib.axes import Axes

mpl.rcParams.update({
  "font.size": 16,
  "axes.titlesize": 20,
  "axes.labelsize": 16,
  "xtick.labelsize": 9,
  "ytick.labelsize": 9,
  "figure.dpi": 300,
  "savefig.dpi": 300,
  "pdf.fonttype": 42,
  "ps.fonttype": 42,
  "mathtext.fontset": "dejavusans",
  "text.usetex": True,
  # "font.family": "Helvetica",
  "font.family": "Computer Modern Roman",
})

In [ ]:
# ── Parameters ──────────────────────────────
# L      = 3
# shape  = (L, L)
N      = 4
n_states = 1 << N

J      = 0.2
h      = 0.0
T1, T2 = 1/3.0, 1/1.0
betas_nu = jnp.array([1/T1, 1/T2])
gammas_nu = jnp.array([1.0, 1.0])
dt = 1e-3

# speed limit eps
eps = 0.05
w1_num_points = 200  # logarithmically spaced time points

# graph = ising.make_interaction_graph(shape)
graph = ising.make_mean_field_graph(N)

N_max = 20_000

# ── Initial condition ─────────────────────────────────────────────────────────
# delta function — all spins up (larger KL, more interesting dynamics)
all_up_idx = (1 << N) - 1
p0 = jnp.zeros(n_states)
p0 = p0.at[all_up_idx].set(1.0)
# p0 = jnp.ones((1 << N)) / (1 << N)

In [ ]:
sys_reduced = ising.build_mean_field_reduced_system(N, J, h, betas_nu, gammas_nu)

In [ ]:
def lift(P):
  return

In [ ]:
import numpy as np
import jax.numpy as jnp
from jax.scipy.linalg import expm

from utils import H
import ising


def check_mean_field_reduction(
  N=4,
  J=0.7,
  h=0.2,
  betas=(3.0, 1.0),
  gammas=(1.0, 0.6),
  t=0.37,
):
  betas = jnp.asarray(betas)
  gammas = jnp.asarray(gammas)

  # Full 2**N-state model.
  graph = ising.make_mean_field_graph(N)
  K_full, g_full = ising.build_system_from_graph(
      graph, J, h, betas, gammas
  )

  # Reduced (N+1)-sector model.
  K_reduced, g_reduced, _, log_deg = (
      ising.build_mean_field_reduced_system(
          N, J, h, betas, gammas
      )
  )

  # sector[sigma] = number of up spins in microstate sigma.
  _, spins = ising.make_state_space_graph(N)
  sector = jnp.asarray(
      (spins.sum(axis=0) + N) / 2,
      dtype=jnp.int32,
  )

  def lift(P):
    """
    Convert sector probabilities P[k] to a full distribution
    that is uniform within each sector.
    """
    return P[sector] * jnp.exp(-log_deg[sector])

  def aggregate(p):
    """Sum a full distribution over magnetization sectors."""
    return jnp.zeros(N + 1, dtype=p.dtype).at[sector].add(p)

  # Use a nontrivial probability distribution that tests every sector.
  P = jnp.arange(1, N + 2, dtype=K_reduced.dtype)
  P = P / P.sum()
  p = lift(P)

  # Check sector degeneracies.
  measured_deg = np.bincount(
      np.asarray(sector),
      minlength=N + 1,
  )
  expected_deg = np.exp(np.asarray(log_deg))

  # Check that lifting and aggregating are inverses.
  P_reconstructed = aggregate(p)

  # Infinitesimal generator check:
  # K_full E P = E K_reduced P.
  dp_full = K_full @ p
  dp_reduced_lifted = lift(K_reduced @ P)

  # Instantaneous bath-resolved heat rates.
  heat_full = g_full @ p
  heat_reduced = g_reduced @ P

  # Full microstate entropy, including sector degeneracy.
  entropy_full = H(p)
  entropy_reduced = ising.mean_field_micro_entropy(
      P, log_deg
  )

  # Finite-time trajectory check.
  p_full_t = expm(K_full * t) @ p
  P_reduced_t = expm(K_reduced * t) @ P
  p_reduced_t_lifted = lift(P_reduced_t)

  # Float32 calculations naturally accumulate errors around 1e-6.
  options = dict(rtol=2e-5, atol=2e-6)

  np.testing.assert_allclose(
      measured_deg, expected_deg, **options
  )
  np.testing.assert_allclose(
      P_reconstructed, P, **options
  )
  np.testing.assert_allclose(
      dp_full, dp_reduced_lifted, **options
  )
  np.testing.assert_allclose(
      heat_full, heat_reduced, **options
  )
  np.testing.assert_allclose(
      entropy_full, entropy_reduced, **options
  )
  np.testing.assert_allclose(
      p_full_t, p_reduced_t_lifted, **options
  )

  errors = {
      "lift/aggregate": float(
          jnp.max(jnp.abs(P_reconstructed - P))
      ),
      "generator": float(
          jnp.max(jnp.abs(dp_full - dp_reduced_lifted))
      ),
      "heat": float(
          jnp.max(jnp.abs(heat_full - heat_reduced))
      ),
      "entropy": float(
          jnp.abs(entropy_full - entropy_reduced)
      ),
      "finite-time trajectory": float(
          jnp.max(
              jnp.abs(p_full_t - p_reduced_t_lifted)
          )
      ),
  }

  print("Mean-field reduction checks passed")
  for name, error in errors.items():
      print(f"{name:24s}: {error:.3e}")

  return errors


errors = check_mean_field_reduction()

In [ ]:
@jax.jit
def reduced_heatmap_sweep(J_values, beta2_values):

    def one_J(J):

        def one_beta(beta2):
            betas = jnp.stack((beta1, beta2))

            pmmc_curve, ep_curve = (
                ising.mean_field_reduced_curves(
                    P0_reduced,
                    N,
                    J,
                    h_reduced,
                    betas,
                    gammas_nu,
                    dt_reduced,
                    N_max,
                )
            )

            # Return only the four horizons used by the heatmap.
            return (
                pmmc_curve[N_idx_jax],
                ep_curve[N_idx_jax],
            )

        return lax.map(one_beta, beta2_values)

    return lax.map(one_J, J_values)

In [ ]:
pmmc_by_J, ep_by_J = reduced_heatmap_sweep(
    J_vals,
    beta_vals,
)

# Wait explicitly because JAX dispatch is asynchronous.
pmmc_by_J.block_until_ready()

# JAX output shape:
# (n_J, n_beta, n_plot)
#
# Heatmap cube shape:
# (n_beta, n_J, n_plot)
pmmc_cube = np.asarray(
    jax.device_get(pmmc_by_J)
).transpose(1, 0, 2)

ep_cube = np.asarray(
    jax.device_get(ep_by_J)
).transpose(1, 0, 2)

print("pmmc_cube:", pmmc_cube.shape)
print("ep_cube:", ep_cube.shape)